<div style="color:#0044aa; text-align:center"><h1>Usando o modelo IGRF (International Geomagnetic Reference Field)</h1>
<p style="text-align: center; font-size:200%">Para obtenção de parâmetros do campo magnético da Terra</p>
</div>
<div style="text-align: center;font-size: 90%;">
    André Ricardo Fasanaro Martinon<sup>1</sup> <sup><a href="https://orcid.org/0000-0002-6578-0170"><i class="fab fa-lg fa-orcid" style="color: #a6ce39"></i></a></sup>, Eurico Rodrigues de Paula<sup>2</sup> <sup><a href="https://orcid.org/0000-0003-2756-3826"><i class="fab fa-lg fa-orcid" style="color: #a6ce39"></i></a></sup>
    <br/><br/>
    <sup>1</sup>Coordenação de Pesquisa Aplicada e Desenvolvimento Tecnológico, <sup>2</sup>Divisão de Heliofísica, Ciências Planetárias e Aeronomia
    <br/>
    <b>Instituto Nacional de Pesquisas Espaciais (INPE)</b>
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brasil
    <br/><br/>
    Contato: <a href="mailto:andre.martinon@inpe.br\">andre.martinon@inpe.br</a>, 
    <a href="mailto:eurico.paula@inpe.br\">eurico.paula@inpe.br</a>
    <br/><br/>
    Última atualização: 28 de Setembro de 2026
</div>
<br/>
<div style="text-align: justify;  margin-left: 15%; margin-right: 15%;">
<span style="color:#0044aa"><h2 style="display: inline">Resumo</h2></span>Este Jupyter Notebook apresenta a biblioteca ppigrf, escrita em Python. Essa biblioteca implementa o modelo IGRF-14 na sua versão atual, e permite calcular o campo magnético da Terra através dos parâmetros fornecidos ao seus métodos. São propostos alguns exercícios onde pode-se estudar o uso da biblioteca ppigrf.
</div>

<div style="color:#0044aa"><h2>Instalar as bibliotecas necessárias</h2></div>
<hr style="border:0; height:1px; background-color:#666666">

### Biblioteca ppigrf (Pure Python International Geomagnetic Reference Field)

Biblioteca escrita em Python que permiter calcular previsões do modelo IGRF. O IGRF é um modelo do campo magnético principal da Terra que é atualizado a cada 5 anos. A versão atual do ppigrf emprega o IGRF-14, atualizado em 2025, em substituição ao IGRF-13 de 2020. Essa biblioteca depende das bibliotecas numpy e pandas.

Mais detalhes sobre a bibliteca pode ser conferido em seu repositório no GitHub: [https://github.com/IAGA-VMOD/ppigrf](https://github.com/IAGA-VMOD/ppigrf)

In [ ]:
#!pip install ppigrf numpy pandas matplotlib cartopy

In [ ]:
import numpy as np
import pandas as pd
import ppigrf
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime

<div style="color:#0044aa"><h2>Exercícios</h2></div>
<hr style="border:0; height:1px; background-color:#666666">

<div style="color:#0044aa"><h3>1. Utilizando o modelo de campo magnético IGRF:</h3></div>

<hr style="border:0; height:1px; background-color:#666666">

**a)** Calcular os parâmetros: **B**, **H**, **X**, **Y**, **Z**, **I**, **D** para **1.2**, **1.4**, **1.6**, **1.8** e **2** vezes o raio da Terra (RT = 6370 km). Considerar a localização geográfica de São Luís/MA (**2° 31' 45" S**, **44° 18' 10" W**) e a data de **30/01/2024**.
<hr style="border:0; height:1px; background-color:#666666">

Primeiro precisamos declarar as variáveis que serão utilizadas nos cálculos:

In [ ]:
RT = 6370           # Raio da Terra em km
lat_sl = -2.529167  # Latitude de São Luís/MA
lon_sl = -44.302778 # Longitude de São Luís/MA

current_date = datetime(2024, 1, 30, 0, 0) # 30/01/2024
size_factors = [1.2, 1.4, 1.6, 1.8, 2]     # Fatores para incrementos de altitude

> ![figura 1](https://thumb.wikimedia.org/wikipedia/commons/thumb/1/16/XYZ-DIS_magnetic_field_coordinates.svg/3840px-XYZ-DIS_magnetic_field_coordinates.svg.png?utm_source=pt.wikipedia.org&utm_campaign=index&utm_content=thumbnail)

> Em qualquer local, o campo magnético da Terra pode ser representado por um vetor tridimensional. Um procedimento típico para medir sua direção é usar uma bússola para determinar a direção do norte magnético. Seu ângulo em relação ao norte verdadeiro é a declinação (D). Diante do norte magnético, o ângulo que o campo faz com a horizontal é a inclinação (I) DIP magnético. A intensidade (B) do campo é proporcional à força que exerce sobre um ímã. Outra representação comum está nas coordenadas X (norte), Y (leste) e Z (abaixo).

Adaptado de: [Wikipedia](https://pt.wikipedia.org/wiki/Campo_magn%C3%A9tico_terrestre)

Agora podemos calcular as componentes do campo magnético e os ângulos de inclinação e declinação de acordo com cada fator de incremento da altitude:

In [ ]:
results = []
for size_factor in size_factors:
    # Altitude
    height = size_factor * RT
 
    # Retorna as componentes do campo magnético de acordo com os parâmetros fornecidos
    Be, Bn, Bu = ppigrf.igrf(lon_sl, lat_sl, h=height, date=current_date)

    X = Bn  # Componente Norte do campo magnético
    Y = Be  # Componente Leste do campo magnético
    Z = -Bu # Componente Vertical do campo magnético (positivo em direção ao centro da Terra)
    
    B = np.linalg.norm([X.item(), Y.item(), Z.item()])         # Intensidade do campo magnético
    H = np.sqrt(np.power(X.item(), 2) + np.power(Y.item(), 2)) # Componente Horizontal do campo magnético (plano X-Y)
    I, D = ppigrf.get_inclination_declination(Be, Bn, Bu)      # Ângulos de Inclinação e Declinação Magnética

    results.append({
        'B (nT)': B, 
        'H (nT)': H, 
        'X (nT)': X.item(), 
        'Y (nT)': Y.item(), 
        'Z (nT)': Z.item(), 
        'I (degrees)': I.item(), 
        'D (degrees)': D.item()
    })

# Compila os resultados em uma tabela
results_table = pd.DataFrame(results, index=size_factors)
results_table

Como podemos observar através dos resultados obtidos, o campo magnético (B) diminui rapidamente com a altitude. O campo magnético é gerado no núcleo externo da Terra (efeito dínamo) e se comporta, de forma aproximada, como um dipolo magnético clássico. Por causa dessa natureza dipolar, a intensidade do campo magnético diminui com o cubo da distância em relação ao centro da Terra: $$\left( \dfrac{1}{r} \right)^3$$

Por outro lado, a inclinição (I) e a declinação (D) magnéticas, não sofrem mudanças significativas, pois a geometria do campo magnético não se altera significativamente com a altitude. 

<hr style="border:0; height:1px; background-color:#666666">

**b)** A razão da declinação em São Luís/MA na superfície da Terra em relação a declinação nas seguintes localizações geográficas, também na superfície da Terra:

1. (**2° 31' 45" S**, **54° 18' 10" W**)
2. (**2° 31' 45" S**, **64° 18' 10" W**)
3. (**2° 31' 45" S**, **74° 18' 10" W**)

Considerar a mesma data da questão anterior.
<hr style="border:0; height:1px; background-color:#666666">

Primeiro precisamos declarar as variáveis que serão utilizadas nos cálculos:

In [ ]:
height = 0          # Altitude
lat_sl = -2.529167  # Latitude de São Luís/MA
lon_sl = -44.302778 # Longitude de São Luís/MA

lat_local1 = -2.529167  # Latitude da localização 1
lon_local1 = -54.302778 # Longitude da localização 1

lat_local2 = -2.529167  # Latitude da localização 2
lon_local2 = -64.302778 # Longitude da localização 2

lat_local3 = -2.529167  # Latitude da localização 3
lon_local3 = -74.302778 # Longitude da localização 3

current_date = datetime(2024, 1, 30, 0, 0) # 30/01/2024

A declinação magnética em **São Luís/MA** será de:

In [ ]:
# Calcular as componentes do campo magnético de acordo com os parâmetros fornecidos para São Luís/MA
Be, Bn, Bu = ppigrf.igrf(lon_sl, lat_sl, h=height, date=current_date)
_, D_sl = ppigrf.get_inclination_declination(Be, Bn, Bu)

D_sl.item()

A declinação magnética na **localização 1** será de:

In [ ]:
# Calcular as componentes do campo magnético de acordo com os parâmetros fornecidos para a localização 1
Be, Bn, Bu = ppigrf.igrf(lon_local1, lat_local1, h=height, date=current_date)
_, D_local1 = ppigrf.get_inclination_declination(Be, Bn, Bu)

D_local1.item()

**A razão entre as declinações magnéticas de São Luís/MA e a primeira localização geográfica será de:**

In [ ]:
# Calcular a razão entre as duas declinações
ratio_local1 = D_sl.item()/D_local1.item()
ratio_local1

A declinação magnética na **localização 2** será de:

In [ ]:
# Calcular as componentes do campo magnético de acordo com os parâmetros fornecidos para a localização 2
Be, Bn, Bu = ppigrf.igrf(lon_local2, lat_local2, h=height, date=current_date)
_, D_local2 = ppigrf.get_inclination_declination(Be, Bn, Bu) 

D_local2.item()

**A razão entre as declinações magnéticas de São Luís/MA e a segunda localização geográfica será de:**

In [ ]:
# Calcular a razão entre as duas declinações
ratio_local2 = D_sl.item()/D_local2.item()
ratio_local2

A declinação magnética na **localização 3** será de:

In [ ]:
# Calcular as componentes do campo magnético de acordo com os parâmetros fornecidos para a localização 3
Be, Bn, Bu = ppigrf.igrf(lon_local3, lat_local3, h=height, date=current_date)
_, D_local3 = ppigrf.get_inclination_declination(Be, Bn, Bu)

D_local3.item()

**A razão entre São Luís/MA e a terceira localização geográfica será de:**

In [ ]:
# Calcular a razão entre as duas declinações
ratio_local3 = D_sl.item()/D_local3.item()
ratio_local3

Podemos observar através dos resultados que a declinação torna-se menos negativa a oeste. Portanto podemos dizer que o Norte magnético desloca-se do Norte geográfico de acordo com a longitude. A declinação tem uma grande dependência em relação a longitude, enquanto a
intensidade do campo magnético tem uma menor variação.

<hr style="border:0; height:1px; background-color:#666666">

**c)** Calcular os parâmetros: **B**, **H**, **X**, **Y**, **Z**, **I**, **D** na superfície da Terra, das seguintes localizações geográficas:

1. (**2° 31' 45" S**, **44° 18' 10" W**)
2. (**12° 31' 45" S**, **44° 18' 10" W**)
3. (**22° 31' 45" S**, **44° 18' 10" W**)
4. (**32° 31' 45" S**, **44° 18' 10" W**)
5. (**42° 31' 45" S**, **44° 18' 10" W**)
6. (**52° 31' 45" S**, **44° 18' 10" W**)

Considerar a mesma data da questão **a)**.
<hr style="border:0; height:1px; background-color:#666666">

Primeiro precisamos declarar as variáveis que serão utilizadas nos cálculos:

In [ ]:
height = 0                   # Altitude
locations = [
    (-44.302778, -2.529167), # Localizações geográficas (lon, lat)
    (-44.302778, -12.529167), 
    (-44.302778, -22.529167),
    (-44.302778, -32.529167),
    (-44.302778, -42.529167),
    (-44.302778, -52.529167)
]
current_date = datetime(2024, 1, 30, 0, 0) # 30/01/2024

Agora podemos calcular as componentes do campo magnético e os ângulos de inclinação e declinação de acordo com cada fator de incremento da altitude:

In [ ]:
results = []
index = []
for lon, lat in locations:
    # Retorna as componentes do campo magnético de acordo com os parâmetros fornecidos
    Be, Bn, Bu = ppigrf.igrf(lon, lat, h=height, date=current_date)

    X = Bn  # Componente Norte do campo magnético
    Y = Be  # Componente Leste do campo magnético
    Z = -Bu # Componente Vertical do campo magnético (positivo em direção ao centro da Terra)
    
    B = np.linalg.norm([X.item(), Y.item(), Z.item()])         # Intensidade do campo magnético
    H = np.sqrt(np.power(X.item(), 2) + np.power(Y.item(), 2)) # Componente Horizontal do campo magnético (plano X-Y)
    I, D = ppigrf.get_inclination_declination(Be, Bn, Bu)      # Ângulos de Inclinação e Declinação Magnética

    results.append({
        'B (nT)': B, 
        'H (nT)': H, 
        'X (nT)': X.item(), 
        'Y (nT)': Y.item(), 
        'Z (nT)': Z.item(), 
        'I (degrees)': I.item(), 
        'D (degrees)': D.item()
    })
    index.append(np.trunc(lat))

# Compila os resultados em uma tabela
results_table = pd.DataFrame(results, index=index)
results_table

Através dos resultados obtidos podemos observar uma diminuição da intensidade do campo magnético até a latitude de 32°S. Depois a intensidade começa a aumentar novamente. Isto ocorre devido a Anomalia do Atlântico Sul.

<div style="color:#0044aa"><h3>2. Utilize o modelo de campo magnético IGRF para determinar os pontos do equador magnético (dip equator) entre as longitudes 30°W e 80°W e latitudes 40°S e 10°N com uma resolução de 0.5° em latitude e longitude. Faça isso para os anos 2009 e 2024 e considere a altitude na superfície da Terra ($altitude=0$). Então trace o equador magnético desses dois anos no mapa da América do Sul.</h3></div>

Primeiramente, precisamos carregar as bibliotecas necessárias para a construção do mapa.

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeatures
import matplotlib as mpl
from matplotlib.figure import Figure
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from cartopy.mpl.ticker import (LongitudeFormatter, LatitudeFormatter)

A função create_world_map, realiza a construção do mapa, exibindo os países e estados de cada continente.

In [ ]:
def create_world_map(ax, extent=(-180, 180, -90, 90)):
    ax.set_extent(extent)

    land = cfeatures.LAND.with_scale('50m')
    ax.add_feature(land, facecolor='none', edgecolor='gray', linewidth=0.5)

    countries = cfeatures.BORDERS.with_scale('50m')
    ax.add_feature(countries, edgecolor='gray', linestyle='-', linewidth=0.5)

    states = cfeatures.STATES.with_scale('50m')
    ax.add_feature(states, edgecolor='gray', linestyle='-', linewidth=0.5)

    gl = ax.gridlines(draw_labels=True, linewidth=1, color='gray', linestyle=(0, (1, 5)))
    gl.top_labels = False
    gl.right_labels = False

    gl.xformatter = LongitudeFormatter(direction_label=False)
    gl.xlocator = mticker.MaxNLocator(nbins=9, steps=[1, 2, 5, 10])
    gl.xlabel_style = {'size': 10, 'weight': 'normal'}

    gl.yformatter = LatitudeFormatter(direction_label=False)
    gl.ylocator = mticker.MaxNLocator(nbins=9, steps=[1, 2, 5, 10])
    gl.ylabel_style = {'size': 10, 'weight': 'normal'}

    gl.xlines = False
    gl.ylines = False

Já a função plot_dip_equator traça o equador magnético.

Para calcular as latitudes magnéticas precisamos transformar a inclinação magnética utilizando a seguinte fórmula:
$$\lambda_m = arctan \left( \dfrac{tan(I)}{2} \right)$$

In [ ]:
def plot_dip_equator(ax, igrf_date, height=0, extent=(-180, 180, -90, 90), resolution=0.5, color='red'):
    lon_min, lon_max, lat_min, lat_max = extent

    lat = np.arange(lat_min, lat_max + resolution, resolution, dtype='float')
    lon = np.arange(lon_min, lon_max + resolution, resolution, dtype='float')

    grid_lon, grid_lat = np.meshgrid(lon, lat)

    Be, Bn, Bu = ppigrf.igrf(grid_lon,
                             grid_lat,
                             h=height,
                             date=igrf_date)
    inclination, declination = ppigrf.get_inclination_declination(Be, Bn, Bu)

    table = np.rad2deg(np.arctan(np.tan(np.deg2rad(inclination[0])) * 0.5))

    contour = ax.contour(grid_lon,
                         grid_lat,
                         table,
                         levels=[0],
                         colors=color,
                         alpha=1,
                         linewidths=1,
                         zorder=2)

    return contour

A seguir o mapa completo é construído:

In [ ]:
extent = [-80, -30, -40, 10]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': ccrs.PlateCarree()})
create_world_map(ax, extent)

cs1 = plot_dip_equator(ax, datetime(2009, 1, 1), extent=extent, color='orange')
cs2 = plot_dip_equator(ax, datetime(2024, 1, 1), extent=extent, color='blue')

artists1, _ = cs1.legend_elements()
artists2, _ = cs2.legend_elements()
_ = plt.legend([artists1[0], artists2[0]], ['2009', '2024'])

A figura mostra um deslocamento do equador magnético para oeste entre os anos de 2009 e 2024. Este movimento não é uniforme ao longo do equador.